# Human-in-the-Loop: Pintu Pra-Tindakan, Penentuan Risiko, dan Log Audit

README untuk pelajaran ini memperkenalkan Human-in-the-Loop dengan petikan pendek yang meminta pengguna untuk `SETUJU` atau `TOLAK` setelah ejen menghasilkan jawapan. Corak itu adalah titik mula yang baik, tetapi pelaksanaan HITL dalam produksi biasanya memerlukan tiga elemen tambahan:

1. **Pintu pra-tindakan** yang dijalankan **sebelum** ejen melaksanakan langkah berisiko, supaya kos, ketidakbolehbalikan, dan kelewatan kekal terkawal.
2. **Penentuan risiko**, supaya tindakan berisiko rendah dilaksanakan secara automatik, tindakan risiko sederhana diluluskan secara berkumpulan, dan hanya tindakan berisiko tinggi yang memerlukan pengesahan manusia.
3. **Log audit dan kitaran semakan**, supaya setiap keputusan pintu direkod sebagai JSONL, dan penolakan akan mengulangi pemanggilan ejen dengan sebab yang terstruktur dan bukannya hanya memaparkan `Menyemak semula...`.

Notebook ini membina setiap elemen tersebut menggunakan primitif yang sama seperti `06-system-message-framework.ipynb`. Ia dijalankan secara menyeluruh dalam `DEMO_MODE = True` (tidak memerlukan input interaktif) atau dengan arahan `input()` sebenar apabila `DEMO_MODE = False`. Nota: dalam DEMO_MODE cubaan semula bagi matlamat ketiga adalah skrip jadi mekanik gelung dapat dilihat secara menyeluruh. Pengelasan semula berasaskan semakan sebenar memerlukan `DEMO_MODE = False` dan seorang operator.

**Di luar skop (ditangani dalam pelajaran lain):** pengesahan dan kawalan akses (ancaman README Pelajaran 06 bahagian 2), middleware panggilan alat (penerokaan mendalam MAF Pelajaran 14), corak perdebatan agen berganda.

In [ ]:
import json
import os
from datetime import datetime, timezone
from pathlib import Path

from dotenv import load_dotenv
from azure.ai.inference import ChatCompletionsClient
from azure.ai.inference.models import SystemMessage, UserMessage
from azure.core.credentials import AzureKeyCredential

load_dotenv()

DEMO_MODE = True  # set False to use real input() prompts

# Per-run unique log filename so demo runs don't overwrite each other and
# the notebook doesn't touch any pre-existing gate_log.jsonl in the working
# directory.
GATE_LOG_PATH = Path(
    f"gate_log_{datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ')}.jsonl"
)

token = os.environ.get("GITHUB_TOKEN", "")
if not token:
    raise RuntimeError(
        "GITHUB_TOKEN environment variable is not set. This notebook needs "
        "a GitHub PAT with 'Models: Read-only' permission. Set GITHUB_TOKEN "
        "in your environment or a local .env file before running."
    )

endpoint = "https://models.github.ai/inference"
model_name = "gpt-4o"

client = ChatCompletionsClient(
    endpoint=endpoint,
    credential=AzureKeyCredential(token),
)


## Corak 1: Gerbang pra-tindakan

Petikan HITL dalam README memanggil agen terlebih dahulu, kemudian meminta pengguna untuk meluluskan output. Itu adalah aliran **pasca-tindakan**. Agen telah melaksanakan, jadi kos panggilan LLM sudah dibayar, dan sebarang kesan sampingan (emel dihantar, baris pangkalan data ditulis, komen diposkan) sudah berlaku.

Aliran **pra-tindakan** meletakkan gerbang sebelum agen menjalankan langkah berisiko. Agen mencadangkan tindakan, gerbang memutuskan sama ada untuk melaksanakan, dan hanya dengan kelulusan barulah kesan sampingan berlaku.

| Aspek | Kelulusan pasca-tindakan (petikan README) | Gerbang pra-tindakan (buku nota ini) |
|---|---|---|
| Bila kelulusan dijalankan? | Selepas agen menghasilkan output | Sebelum sebarang kesan sampingan dilaksanakan |
| Kos LLM pada penolakan | Sudah dibayar | Dibayar hanya untuk cadangan, bukan tindakan |
| Kesan sampingan yang tidak boleh dipulihkan | Mungkin (tindakan sudah berlaku) | Dihalang |
| Kejelasan audit | Kelulusan adalah kenyataan cetak | Kelulusan adalah rekod JSON dengan cap masa, tindakan, sebab |


In [ ]:
def gate_action(action_description: str, risk_tier: str, attempt: int = 0) -> dict:
    """Run a single pre-action gate.

    Returns a decision dict with keys: decision, reason, ts.
    Decision is one of: approve, deny, escalate.
    Safe default on EOF or unexpected input is deny.

    DEMO_MODE behavior: high-risk actions are denied on attempt 0 and
    auto-approved on attempt >= 1. This is scripted approval to show the
    loop mechanics (deny -> retry -> approve). It is NOT revision-driven
    re-classification. Real revision-driven re-classification requires
    DEMO_MODE=False and a human operator who evaluates the revised
    proposal on its own merits.
    """
    print(f"[gate] proposed action ({risk_tier}, attempt={attempt}): {action_description}")

    if DEMO_MODE:
        if risk_tier == "high":
            decision = "approve" if attempt >= 1 else "deny"
            reason = (
                "DEMO_MODE: scripted approval on retry to show loop mechanics"
                if attempt >= 1
                else "DEMO_MODE: high risk denied on first attempt"
            )
        else:
            decision = "approve"
            reason = f"DEMO_MODE canned response for tier={risk_tier}"
    else:
        try:
            raw = input("[gate] approve / deny / escalate? ").strip().lower()
        except EOFError:
            raw = ""
        if raw in {"approve", "deny", "escalate"}:
            decision, reason = raw, "operator input"
        elif raw == "":
            decision, reason = "deny", "no input received, defaulted to deny"
        else:
            decision, reason = "deny", f"invalid input {raw!r}, defaulted to deny"

    return {
        "decision": decision,
        "reason": reason,
        "action": action_description,
        "risk_tier": risk_tier,
        "ts": datetime.now(timezone.utc).isoformat(),
    }


## Corak 2: Penentuan risiko mengikut tahap

Tidak setiap tindakan memerlukan kelulusan manusia. Carian baca-sahaja terhadap API awam mempunyai pertaruhan yang berbeza daripada menghantar emel kepada pelanggan. Melayan kedua-duanya sama sahaja membazirkan perhatian operator dan memperlahankan agen.

Model 3-tahap yang mudah:

| Tahap | Contoh | Aliran kelulusan |
|---|---|---|
| `rendah` (baca-sahaja) | Cari pangkalan pengetahuan, semak pilihan penerbangan, ambil halaman web awam | Laksana auto, direkodkan untuk audit |
| `sederhana` (mutasi murah) | Cache hasil, tambah pengira, jadualkan peringatan | Laksana auto, tetapi disemak secara berkumpulan harian |
| `tinggi` (berhadapan luar atau tidak boleh diundur) | Hantar emel, caj kad, pos ke saluran awam | Tangguh atas kelulusan manusia |

Ini adalah satu penentuan tahap. Sistem produksi sering menggunakan tahap yang lebih terperinci (contohnya, tahap kebenaran AWS IAM, tahap akses berdasarkan peranan). Versi 3-tahap di bawah adalah versi paling kecil yang berguna untuk agen yang menggabungkan tindakan baca-sahaja dan yang mempunyai kesan sampingan.

Pengelasan di bawah menggunakan heuristik kata kunci supaya demo kekal deterministik dan murah. Dalam sistem produksi, anda boleh gantikan ini dengan pengelasan yang dipelajari atau enjin dasar.

In [ ]:
LOW_RISK_KEYWORDS = {
    "look", "lookup", "search", "fetch", "read", "query", "view",
    "get", "list", "weather", "summarize",
}
HIGH_RISK_KEYWORDS = {
    "send", "email", "post", "publish", "charge", "pay", "transfer",
    "delete", "drop", "cancel", "refund",
}
MEDIUM_RISK_KEYWORDS = {
    "cache", "schedule", "reminder", "book", "reserve", "update",
    "increment", "log",
}

AUTO_APPROVE_REASONS = {
    "low": "auto-approved (low risk)",
    "medium": "auto-approved (medium risk, queued for batched review)",
}


def classify_risk(action: str) -> str:
    """Classify an action string into one of: low, medium, high.

    Keyword-based heuristic. Checks high-risk first (most severe), then
    low-risk explicit reads, then medium-risk mutations. Unrecognized
    actions default to medium, not low.

    Default for unrecognized actions is 'medium', not 'low'. A read-only
    keyword set will always have blind spots, and the parent README's
    threat list (critical-system access, knowledge-base poisoning,
    cascading errors) all involve cases an action-name alone cannot rule
    out. Routing unknown actions through batched review is the safer
    default than auto-executing them.
    """
    text = action.lower()
    if any(kw in text for kw in HIGH_RISK_KEYWORDS):
        return "high"
    if any(kw in text for kw in LOW_RISK_KEYWORDS):
        return "low"
    if any(kw in text for kw in MEDIUM_RISK_KEYWORDS):
        return "medium"
    # Explicit fail-safe default: unrecognized actions route to batched review.
    return "medium"


def tiered_gate(action: str, attempt: int = 0) -> dict:
    """Classify then gate. Low and medium tiers auto-approve; high blocks."""
    tier = classify_risk(action)
    if tier in AUTO_APPROVE_REASONS:
        return {
            "decision": "approve",
            "reason": AUTO_APPROVE_REASONS[tier],
            "action": action,
            "risk_tier": tier,
            "ts": datetime.now(timezone.utc).isoformat(),
        }
    return gate_action(action, tier, attempt=attempt)


## Corak 3: Log audit dan gelung semakan

Satu `print("Response approved.")` bukanlah log audit. Untuk keyakinan, setiap keputusan pintu pagar harus direkodkan sebagai acara berstruktur yang anda boleh kemudian soal, main semula, atau lampirkan pada semakan insiden.

Dua bahagian:

1. **JSONL hanya tambah.** Satu baris setiap keputusan, dengan cap masa, tindakan, peringkat, keputusan, sebab. Mudah untuk grep, mudah dihantar ke stor log sebenar kemudian.
2. **Gelung semakan pada penolakan.** Apabila pintu pagar mengembalikan `deny`, ejen mengingatkan semula dirinya dengan sebab penolakan dalam konteks, supaya cadangan seterusnya boleh mengelakkan masalah.

In [ ]:
def log_decision(decision: dict) -> None:
    """Append a gate decision to the JSONL audit log."""
    with GATE_LOG_PATH.open("a", encoding="utf-8") as f:
        f.write(json.dumps(decision) + "\n")


def propose_action(goal: str, prior_rejection: str | None = None) -> str:
    """Ask the LLM to propose a concrete next action for a goal.

    If prior_rejection is provided, it is fed back so the LLM can avoid
    the same failure mode in the next proposal.
    """
    system = (
        "You are an action planner for an agent. Propose ONE concrete next\n"
        "action (a single sentence) toward the user's goal. If a prior\n"
        "rejection reason is given, propose a different action that addresses\n"
        "the rejection."
    )
    user_text = f"Goal: {goal}"
    if prior_rejection:
        user_text += f"\n\nPrior proposal was denied. Reason: {prior_rejection}"

    response = client.complete(
        model=model_name,
        messages=[SystemMessage(content=system), UserMessage(content=user_text)],
    )
    return response.choices[0].message.content.strip()


def run_with_revision(goal: str, max_revisions: int = 2) -> dict:
    """Propose, gate, and on rejection revise up to max_revisions times."""
    prior_reason: str | None = None
    for attempt in range(max_revisions + 1):
        action = propose_action(goal, prior_rejection=prior_reason)
        decision = tiered_gate(action, attempt=attempt)
        decision["attempt"] = attempt
        log_decision(decision)
        if decision["decision"] == "approve":
            return decision
        prior_reason = decision["reason"]
    return {**decision, "final": "max_revisions_reached"}


In [ ]:
# End-to-end demo: three goals at three different risk profiles.
# GATE_LOG_PATH is per-run (timestamped) so no prior log is touched.

goals = [
    "Look up the weather in Seattle for the customer's trip planning.",
    "Schedule a reminder for the customer to check in 24 hours before their flight.",
    "Send a marketing email to the customer about premium upgrade options.",
]

for goal in goals:
    print(f"\n=== Goal: {goal} ===")
    outcome = run_with_revision(goal, max_revisions=1)
    print(f"[final] {outcome['decision']} ({outcome['reason']})")

print(f"\n=== Audit log ({GATE_LOG_PATH.name}) ===")
for line in GATE_LOG_PATH.read_text(encoding="utf-8").splitlines():
    record = json.loads(line)
    print(f"  [{record['risk_tier']:6s}] {record['decision']:8s} "
          f"attempt={record.get('attempt', '?')} action={record['action'][:140]}")


## Sumber tambahan

Beberapa projek awam lain melaksanakan variasi corak HITL ini. Bandingkan pendekatan untuk mencari apa yang sesuai dengan set alat anda:

- **LangChain** pembalut alat human-in-the-loop ([dokumen](https://python.langchain.com/docs/integrations/tools/human_tools)): pembalut alat yang boleh dimasukkan terus yang menghentikan pelaksanaan untuk input manusia.
- **AutoGen** `UserProxyAgent` ([dokumen v0.2](https://microsoft.github.io/autogen/0.2/docs/topics/human-in-the-loop); AutoGen v0.4+ mengubah strukturnya): menggunakan peranan agen khusus untuk mewakili manusia dalam perbualan pelbagai agen.
- **Semantic Kernel** penapis fungsi ([dokumen](https://learn.microsoft.com/en-us/semantic-kernel/concepts/enterprise-readiness/filters)): middleware yang dijalankan di sekitar setiap panggilan fungsi, sesuai untuk logik pengawalan.

Setiap projek mengendalikan tiga sub-corak ini secara berbeza: LangChain membalutnya sebagai alat, AutoGen menggunakan peranan agen, Semantic Kernel menggunakan penapis middleware. Baca satu atau dua pelaksanaan secara keseluruhan sebelum memilih reka bentuk untuk agen anda sendiri.

---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Penafian**:
Dokumen ini telah diterjemahkan menggunakan perkhidmatan terjemahan AI [Co-op Translator](https://github.com/Azure/co-op-translator). Walaupun kami berusaha untuk ketepatan, sila ambil maklum bahawa terjemahan automatik mungkin mengandungi kesilapan atau ketidaktepatan. Dokumen asal dalam bahasa asalnya harus dianggap sebagai sumber yang sahih. Untuk maklumat penting, terjemahan oleh manusia profesional adalah disyorkan. Kami tidak bertanggungjawab terhadap sebarang salah faham atau salah tafsir yang timbul daripada penggunaan terjemahan ini.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
